# Variational AutoEncoder For Limb Generation
So heres the idea - the 7 measurements that we have don't fully define the geometry of these limbs. Not even close. Shape and bend and whatnot don't show up in measurement so what we're going to do is build an auto encoder for reconstructing the limb from its coefficients. We condition the decoder section on an embedding (Nonsense numbers) and the measurements instead of just reconstructing from the measurements themselves. That way, we have a few values we can play around with that should generate limbs with the same measurements but varying around the final shape of the limb. Insha'Allah. 

In [ ]:
#| export_section
import torch
from torch import nn
import subprocess
import os
from functools import partial
import numpy as np
import igl
import measure_limbs
from SSM_Driver import Measurements, measurement_details, LegMeasurementDataset, create_examples
#| end_section

from tqdm import tqdm
import wandb

In [ ]:
config = {
    "lr": 1e-3,
    "eta_min": 1e-6,
    "batch_size": 256,
    "log": False,
    "seed": 42,
    "epochs":200,
}

torch.manual_seed(config["seed"])

if config["log"]:
    wandb.init(project="Open Limb SSM", config=config)

In [ ]:
dtype = torch.float64
mean_verts = torch.load("./data_components/mean_verts.pt").to(dtype)
face2vert = torch.load("./data_components/face2vert.pt")

edge2vert, face2edge, edge2face = igl.edge_topology(
    mean_verts.numpy(), face2vert.numpy()
)

edge2vert = torch.from_numpy(edge2vert)
face2edge = torch.from_numpy(face2edge)
edge2face = torch.from_numpy(edge2face)

measure = Measurements(
    edge2vert,
    face2edge,
    measurement_details,
)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input, output):
        # Input: 7 measurements + 10 coefficients = 17
        self.net = nn.Sequential(
            nn.Linear(input, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.mu = nn.Linear(32, output)      # z_dim = 3
        self.logvar = nn.Linear(32, output)

    def forward(self, x):
        x = self.net(x)
        return self.mu(x), self.logvar(x)

class Decoder(nn.Module):
    def __init__(self, input, output):
        # Input: 3 (z) + 7 (measurements) = 10
        self.net = nn.Sequential(
            nn.Linear(input, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, output)  # Output: 10 coefficients
        )

    def forward(self, x):
        return self.net(x)
    
class VAE(nn.Module):
    def __init__(self, encoder_input, z_dim, measurement_dim, decoder_output, measurement):
        self.encoder = Encoder(encoder_input, z_dim)
        self.decoder = Decoder(z_dim + measurement_dim, decoder_output)
        self.measurement = measurement
        self.z_dim = z_dim

    def forward(self, measurements, coefficients):
        mean, log_var = self.encoder(torch.cat((measurements, coefficients)))
        
        # Reparameterisation?
        epsilon = torch.randn_like(mean).to(measurements.device)
        z = mean + torch.exp(0.5*log_var)*epsilon

        x_hat = self.decoder(torch.cat((measurements, z)))

        return x_hat, mean, log_var
    
    def sample(self, measurements, samples=64):
        self.eval()
        with torch.no_grad():
            z = torch.randn(samples, self.z_dim).to(measurements.device)

            # This is definitely not right but Im trying to figure out what I want from it. 
            # Do I want to have (measurements, samples, coeffs) or (samples, coeffs) from a single measurement?
            decoder_input = torch.cat((z, measurements), dim=-1)

            coeffs = self.decoder(decoder_input)




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scale = False
normalise = False
relativise = False

dataset = LegMeasurementDataset(measure, batch_size=config["batch_size"], scale=scale, normalise=normalise, relativise=relativise, dtype=dtype, device="cpu")
dataloader = torch.utils.data.DataLoader(dataset, config["batch_size"], shuffle=False, num_workers=0)
loss_func = nn.MSELoss().to(device)
# loss_func = MeasurementLoss(dataset).to(device)
# loss_func = PointwiseLoss(dataset).to(device)
component_loss = nn.MSELoss()

# Add one for the scale factor 
component_transforms = torch.load("./data_components/scaled_component_transforms.pt")[:,:10+scale].to(dtype).to(device)
model = VAE(17, 3, 7, 10)

optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config["epochs"])

In [ ]:
from SSM_Driver import LegMeasurementDataset as LMD
from SSM_Driver import measure
data = LMD(measure)